# Gaussian in the Bay

In this case, we solve for a Gaussian bump advecting with the currents.

## Formal problem formulation

We want to find depth-averaged concentration $c(x, t)$ as the solution to the depth-averaged advection equation.

$$
\begin{cases}
    \frac{\partial \iota}{\partial t} + \nabla \cdot (\iota \mathbf{v}) = 0 &x \in \Omega \\
    \iota = 0 &x \in \Gamma_{\mathrm{in}}
\end{cases}
$$

- The conserved quantity $\iota$ is concentration scaled by height, $\iota = c h$.
- The domain $\Omega$ is taken to be the whole mesh from example 2, including dry land. This naturally handles wetting/drying, because inactivated cells have no momentum, and the conserved quantity $\iota$, which can be interpreted as salt mass per infinitesimal column, remains fixed as a "salt flat." Concentration $c$, however, is not well-defined and blows up.
- The time range is taken to be 9 days, $(0, T) = (0, 777600)$.
- Inflow boundaries $\Gamma_{\mathrm{in}}$ are prescribed no concentration, $\iota_{\mathrm{in}} = 0$.
- The height $h = \eta + b$ and velocity $v$ fields are derived from fort.63 and fort.64 nodal data files. These values are interpolated both in space (using a Matplotlib triangulation) and time (using a linear interpolation).

$$
\begin{align}
\mathbf{v}(x, t) &= (\mathbf{v}_{b}(x) - \mathbf{v}_{a}(x)) \frac{t - t_{a}}{t_{b} - t_{a}} + \mathbf{v}_{a} \\
\\ &= \frac{t - t_{a}}{t_{b} - t_{a}} \mathbf{v}_{b} + \frac{t_{b} - t}{t_{b} - t_{a}} \mathbf{v}_{a} \\
&t \in [t_{a}, t_{b}]
\end{align}
$$

- Crucually, the metric tensor is not yet implemented. For now, longitude and latitude are treated as Cartesian coordinates.

## Imports and setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import adcircxtools as at
import adios4dolfinx as adx
import basix.ufl
import dolfinx as dx
import dolfinx.fem.petsc
import dolfinx.plot as dxp
import fenicsxtools as ft
import matplotlib.pyplot as plt
import matplotlib.tri
from mpi4py import MPI
import numpy as np
from petsc4py import PETSc
import pyvista as pv
import scipy.interpolate
import scipy.spatial
import ufl

## Problem Parameters

The problem is initialized with a Gaussian bump at the center. Note that this includes initialization on dry land.

In [3]:
# For initial condition
radius_earth = 6371.0 # km
pa_lon = -97.0611 # deg
pa_lat = 27.8339 # deg
x0 = pa_lon + 0.0 # Offset
y0 = pa_lat + 0.0 # Offset
r0 = np.rad2deg(1.0 / radius_earth) # deg
def iota0(x):
    return np.exp(-((x[0] - x0) ** 2 + (x[1] - y0) ** 2)/ (2 * r0 ** 2))

In [4]:
# Initial velocity
# Field at time 0 is (0, 0)
def v0(x):
    return np.zeros((2, x.shape[1]))

In [5]:
# Time stepping parameters
t_final = 9.0 * 86400 # seconds
dt = 10.0 # seconds
write_every = 360 # Snapshot every hour
fps = 30 # For plot gif

## Read in mesh and set up data buffers

In [6]:
# Get main filtered mesh
domain = adx.read_mesh('port_aransas.bp', MPI.COMM_WORLD)

In [7]:
# Get structures for slightly larger mesh
# Used for triangular interpolation
interp_coordinates = np.load('interp_coordinates.npy')
interp_elements = np.load('interp_elements.npy')
interp_node_map = np.load('interp_node_map.npy')
interp_element_map = np.load('interp_element_map.npy')

In [8]:
interp_coordinates.shape

(6687, 2)

In [9]:
# Initialize triangulation for function interpolation
triangulation_interp = matplotlib.tri.Triangulation(
    interp_coordinates[:, 0],
    interp_coordinates[:, 1],
    interp_elements
)

In [10]:
# Boolean masks for filtering out data points
interp_node_mask = interp_node_map != -1
interp_element_mask = interp_element_map != -1

In [13]:
# Set up buffers
elevation_buffer = at.io.TimeSeriesBuffer('6hr.63')
elevation_buffer.open()
velocity_buffer = at.io.TimeSeriesBuffer('6hr.64')
velocity_buffer.open()

In [14]:
# Read in first record for initialization
(elevation_buffer_time,
 elevation_buffer_it,
 elevation_buffer_dat) = elevation_buffer.read_step(interp_node_mask)
(velocity_buffer_time,
 velocity_buffer_it,
 velocity_buffer_dat) = velocity_buffer.read_step(interp_node_mask)

In [15]:
velocity_buffer_dat.shape

(6687, 2)

In [25]:
# Set up interpolation function for initialization
# This has to be re-set every time velocity_buffer_dat is updated
u_velocity_interpolator = matplotlib.tri.LinearTriInterpolator(
    triangulation_interp,
    velocity_buffer_dat[:, 1]
)
v_velocity_interpolator = matplotlib.tri.LinearTriInterpolator(
    triangulation_interp,
    velocity_buffer_dat[:, 1]
)
def vel(x):
    return np.vstack((u_velocity_interpolator(x[0], x[1]), v_velocity_interpolator(x[0], x[1])))

## FEM formulation

In [26]:
# Use piecewise linear elements
V = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1))
W = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1, (domain.geometry.dim,)))

In [27]:
# Initialize scaled concentration iota
iota = dx.fem.Function(V, name='iota')
iota.interpolate(iota0)

In [30]:
# Initialize velocity field function
# Defined as interpolation between two field snapshots
va = dx.fem.Function(W)
vb = dx.fem.Function(W)
va.interpolate(v0)
vb.interpolate(vel)
t = dx.fem.Constant(domain, 0.0) # Current simulation time
ta = dx.fem.Constant(domain, 0.0) # First time in interpolation interval
tb = dx.fem.Constant(domain, velocity_buffer_time) # Second time in interpolation interval
v = (t - ta) / (tb - ta) * vb + (tb - t) / (tb - ta) * va

In [31]:
# Generate conservation law
equation = ft.equations.get_advection(
    domain=domain,
    U=iota,
    v=v
)

In [32]:
# Choose standard DG weak formulation
# Use LLF flux
formulation = ft.formulations.DGFormulation(
    equation=equation,
    trace_function=ft.fluxes.fluxn_llf_scalar
)

## Solver formulation

In [ ]:
# Build TS solver
# Plotter
# Function updater

## Solve

## Visualization

## Cleanup

In [ ]:
# Close buffers
elevation_buffer.close()
velocity_buffer.close()